In [1]:
!pip install -q transformers numpy tqdm

import os
import json
import glob
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from pathlib import Path

In [2]:
INPUT_DIR = Path('/content/drive/MyDrive/data-filtering')
OUTPUT_DIR = INPUT_DIR / "bin"
TOKENIZER_NAME = "Qwen/Qwen3.5-0.8B"

BATCH_SIZE = 1e4                     # Размер батча для токенизации
TOKENS_PER_FILE = 1e5          # ~50M токенов на файл (~200MB в uint32)
MAX_FILES = None                      # None = обработать всё, int = лимит файлов для теста

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📂 Input: {INPUT_DIR} | Output: {OUTPUT_DIR}")
print(f"🔤 Tokenizer: {TOKENIZER_NAME}")

📂 Input: /content/drive/MyDrive/data-filtering | Output: /content/drive/MyDrive/data-filtering/bin
🔤 Tokenizer: Qwen/Qwen3.5-0.8B


In [3]:
# %%
def load_text_batches(input_dir, batch_size):
    """Потоковый читатель JSONL/TXT без загрузки всего датасета в память."""
    files = sorted(glob.glob(os.path.join(input_dir, "*.jsonl")) +
                   glob.glob(os.path.join(input_dir, "*.txt")))
    if not files:
        raise FileNotFoundError(f"No .jsonl or .txt files found in {input_dir}")

    for filepath in files:
        with open(filepath, "r", encoding="utf-8") as f:
            batch = []
            for line in f:
                line = line.strip()
                if not line: continue

                if filepath.endswith(".jsonl"):
                    try:
                        data = json.loads(line)
                        text = data.get("text", "")
                    except json.JSONDecodeError:
                        continue
                else:
                    text = line

                if text.strip():
                    batch.append(text)
                if len(batch) >= batch_size:
                    yield batch
                    batch = []
            if batch:
                yield batch

In [4]:
def convert_to_bin(input_dir, output_dir, tokenizer_name,
                   batch_size=2000, tokens_per_file=50_000_000, max_files=None):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Проверка вокабуляра для выбора dtype
    vocab_size = len(tokenizer)
    dtype = np.uint16 if vocab_size <= 65535 else np.uint32
    print(f"📊 Vocab size: {vocab_size:,} → Using dtype: {dtype}")

    buffer = []
    file_idx = 0
    total_tokens = 0
    meta = {"files": [], "tokens_per_file": tokens_per_file, "tokenizer": tokenizer_name, "dtype": str(dtype)}

    pbar = tqdm(load_text_batches(input_dir, batch_size), desc="Tokenizing & Saving")

    for batch in pbar:
        # Быстрая пакетная токенизация
        enc = tokenizer(batch, truncation=False, return_attention_mask=False)

        for ids in enc["input_ids"]:
            buffer.extend(ids)

        # Сброс в .bin при заполнении буфера
        while len(buffer) >= tokens_per_file:
            chunk = buffer[:tokens_per_file]
            buffer = buffer[tokens_per_file:]

            out_path = os.path.join(output_dir, f"data_{file_idx:05d}.bin")
            np.array(chunk, dtype=dtype).tofile(out_path)

            meta["files"].append({"filename": f"data_{file_idx:05d}.bin", "tokens": tokens_per_file})
            total_tokens += tokens_per_file
            file_idx += 1
            pbar.set_postfix({"chunks_saved": file_idx})

            if max_files and file_idx >= max_files:
                pbar.close()
                print(f"🛑 Лимит файлов ({max_files}) достигнут. Остановка.")
                # Сохраняем meta и выходим
                meta["total_tokens"] = total_tokens
                meta["num_files"] = file_idx
                with open(os.path.join(output_dir, "meta.json"), "w") as f:
                    json.dump(meta, f, indent=2)
                return

        # Очистка RAM при переполнении буфера (если что-то пошло не так)
        if len(buffer) > tokens_per_file * 3:
            buffer = buffer[-tokens_per_file:]

    # Сохраняем остаток
    if buffer:
        out_path = os.path.join(output_dir, f"data_{file_idx:05d}.bin")
        np.array(buffer, dtype=dtype).tofile(out_path)
        meta["files"].append({"filename": f"data_{file_idx:05d}.bin", "tokens": len(buffer)})
        total_tokens += len(buffer)
        file_idx += 1

    meta["total_tokens"] = total_tokens
    meta["num_files"] = file_idx

    with open(os.path.join(output_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\n✅ Готово! {total_tokens:,} токенов → {file_idx} файлов в {output_dir}")

In [5]:
convert_to_bin(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    tokenizer_name=TOKENIZER_NAME,
    batch_size=BATCH_SIZE,
    tokens_per_file=TOKENS_PER_FILE,
    max_files=MAX_FILES
)

You are using a model of type qwen3_5 to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


📊 Vocab size: 248,077 → Using dtype: <class 'numpy.uint32'>


Tokenizing & Saving: 0it [00:00, ?it/s]


✅ Готово! 98,582 токенов → 1 файлов в /content/drive/MyDrive/data-filtering/bin
